In [1]:
!pip install --upgrade onnx
!pip install --upgrade onnx onnxruntime
!pip install --upgrade onnx onnx-simplifier
!pip install --upgrade onnx onnx-simplifier onnxruntime

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
"""
Script de quantisation YOLOP optimisé - Sans warnings
Corrige le problème des axes BatchNorm
"""

import os
import numpy as np
import onnx
import onnxruntime
from onnxsim import simplify
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat
from PIL import Image
from torchvision import transforms


In [3]:

# --- CONFIGURATION ---
BASE_DIR = os.getcwd()
INPUT_MODEL_PATH = os.path.join(BASE_DIR, "weights", "yolop-640-640.onnx")
OUTPUT_DIR = os.path.join(BASE_DIR, "weights")
OUTPUT_MODEL_PATH = os.path.join(OUTPUT_DIR, "yolop-640-640-quantized-per_channel.onnx")
CALIBRATION_IMG_DIR = os.path.join(BASE_DIR, "dataset", "images", "val")
CALIBRATION_SIZE = 100
TARGET_SIZE = (640, 640)


In [4]:


class YOLOPDataReader(CalibrationDataReader):
    """DataReader pour la calibration avec preprocessing YOLOP standard"""
    
    def __init__(self, image_folder, model_path, size_limit=100):
        self.image_folder = image_folder
        self.enum_data_dicts = None
        self.datasize = 0
        
        # Récupération du nom de l'entrée du modèle
        session = onnxruntime.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        self.input_name = session.get_inputs()[0].name
        
        # Normalisation ImageNet (standard YOLOP)
        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        files = [f for f in os.listdir(image_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]
        self.image_files = files[:size_limit]
        self.datasize = len(self.image_files)
        print(f"✓ DataReader initialisé avec {self.datasize} images")
    
    def letterbox_image(self, image, size):
        """Redimensionne avec letterbox (garde le ratio)"""
        iw, ih = image.size
        w, h = size
        scale = min(w/iw, h/ih)
        nw = int(iw * scale)
        nh = int(ih * scale)
        
        image = image.resize((nw, nh), Image.BICUBIC)
        new_image = Image.new('RGB', size, (114, 114, 114))
        new_image.paste(image, ((w - nw) // 2, (h - nh) // 2))
        return new_image
    
    def get_next(self):
        if self.enum_data_dicts is None:
            self.enum_data_dicts = iter(self.load_batch())
        try:
            return next(self.enum_data_dicts)
        except StopIteration:
            return None
    
    def load_batch(self):
        for filename in self.image_files:
            image_path = os.path.join(self.image_folder, filename)
            image = Image.open(image_path).convert("RGB")
            image = self.letterbox_image(image, TARGET_SIZE)
            
            input_tensor = self.normalize(image)
            input_numpy = input_tensor.unsqueeze(0).numpy()
            
            yield {self.input_name: input_numpy}



In [5]:

def clean_bn_metadata(model_path, output_path):
    """
    Nettoie les métadonnées BatchNorm problématiques
    Corrige les attributs 'axis' invalides sur les poids BN
    """
    print("→ Nettoyage des métadonnées BatchNorm...")
    
    model = onnx.load(model_path)
    
    # Parcourir tous les initializers (poids)
    for init in model.graph.initializer:
        # Détecter les poids BatchNorm (format: *.bn.weight, *.bn.bias)
        if '.bn.weight' in init.name or '.bn.bias' in init.name:
            # Ces tenseurs sont 1D (rank=1), donc axis ne peut être que 0
            # On force axis=0 si présent dans les attributs
            # Note: en ONNX, les initializers n'ont pas d'attributs directs,
            # le problème vient des nodes qui les utilisent
            pass
    
    # Parcourir les nodes pour corriger les attributs axis
    for node in model.graph.node:
        if node.op_type == 'QuantizeLinear' or node.op_type == 'DequantizeLinear':
            # Vérifier si le node référence un poids BN
            for input_name in node.input:
                if '.bn.weight' in input_name or '.bn.bias' in input_name:
                    # Corriger l'attribut axis s'il existe
                    for attr in node.attribute:
                        if attr.name == 'axis':
                            # Pour les tenseurs 1D BN, axis doit être 0
                            attr.i = 0
    
    onnx.save(model, output_path)
    print(f"✓ Modèle nettoyé : {output_path}")



In [6]:

def quantize_model():
    print("=" * 60)
    print("  QUANTIFICATION YOLOP (Static INT8 PTQ)")
    print("=" * 60)
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if not os.path.exists(INPUT_MODEL_PATH):
        print(f" ERREUR : Modèle introuvable : {INPUT_MODEL_PATH}")
        return
    
    # ============================================================
    # ÉTAPE 1 : SIMPLIFICATION (Fusion Conv+BN)
    # ============================================================
    print("\n[1/4] Simplification du modèle (onnx-simplifier)...")
    SIMPLIFIED_PATH = os.path.join(OUTPUT_DIR, "temp_simplified.onnx")
    
    model_proto = onnx.load(INPUT_MODEL_PATH)
    model_sim, check = simplify(model_proto)
    
    if not check:
        print("❌ Échec de la simplification")
        return
    
    onnx.save(model_sim, SIMPLIFIED_PATH)
    print(f"✓ Modèle simplifié : {SIMPLIFIED_PATH}")
    
    # ============================================================
    # ÉTAPE 2 : NETTOYAGE DES MÉTADONNÉES BN
    # ============================================================
    print("\n[2/4] Correction des métadonnées BatchNorm...")
    CLEANED_PATH = os.path.join(OUTPUT_DIR, "temp_cleaned.onnx")
    clean_bn_metadata(SIMPLIFIED_PATH, CLEANED_PATH)
    
    # ============================================================
    # ÉTAPE 3 : CALIBRATION
    # ============================================================
    print(f"\n[3/4] Préparation des données de calibration...")
    print(f"    Source : {CALIBRATION_IMG_DIR}")
    
    if not os.path.exists(CALIBRATION_IMG_DIR):
        print(f" ERREUR : Dossier de calibration introuvable")
        return
    
    dr = YOLOPDataReader(CALIBRATION_IMG_DIR, CLEANED_PATH, size_limit=CALIBRATION_SIZE)
    
    # ============================================================
    # ÉTAPE 4 : QUANTIFICATION
    # ============================================================
    print(f"\n[4/4] Quantification en cours...")
    print("    Format : QDQ (Quantize-DeQuantize)")
    print("    Activation : QUInt8")
    print("    Weights : QInt8")
    print("    Per-channel : Activé")
    
    try:
        quantize_static(
            model_input=CLEANED_PATH,
            model_output=OUTPUT_MODEL_PATH,
            calibration_data_reader=dr,
            quant_format=QuantFormat.QDQ,
            activation_type=QuantType.QUInt8,
            weight_type=QuantType.QInt8,
            per_channel=True,
            reduce_range=True,
            # Pas besoin de extra_options avec le nettoyage préalable
        )
        
        print("\n" + "=" * 60)
        print("  RÉSULTATS")
        print("=" * 60)
        
        if os.path.exists(OUTPUT_MODEL_PATH):
            size_original = os.path.getsize(INPUT_MODEL_PATH) / (1024 * 1024)
            size_quantized = os.path.getsize(OUTPUT_MODEL_PATH) / (1024 * 1024)
            reduction = size_original / size_quantized
            
            print(f"\n Taille des modèles :")
            print(f"    Original (FP32)  : {size_original:>8.2f} MB")
            print(f"    Quantized (INT8) : {size_quantized:>8.2f} MB")
            print(f"    Réduction        : {reduction:>8.2f}x")
            print(f"\n Modèle quantifié sauvegardé :")
            print(f"    {OUTPUT_MODEL_PATH}")
            
            # Nettoyage des fichiers temporaires
            print(f"\n Nettoyage des fichiers temporaires...")
            for temp_file in [SIMPLIFIED_PATH, CLEANED_PATH]:
                if os.path.exists(temp_file):
                    os.remove(temp_file)
                    print(f"    ✓ Supprimé : {os.path.basename(temp_file)}")
            
        else:
            print(" Erreur : Le fichier de sortie n'a pas été créé")
            
    except Exception as e:
        print(f"\n ERREUR lors de la quantification :")
        print(f"    {str(e)}")
        import traceback
        traceback.print_exc()


In [7]:


if __name__ == "__main__":
    quantize_model()

  QUANTIFICATION YOLOP (Static INT8 PTQ)

[1/4] Simplification du modèle (onnx-simplifier)...
✓ Modèle simplifié : /workspace/weights/temp_simplified.onnx

[2/4] Correction des métadonnées BatchNorm...
→ Nettoyage des métadonnées BatchNorm...
✓ Modèle nettoyé : /workspace/weights/temp_cleaned.onnx

[3/4] Préparation des données de calibration...
    Source : /workspace/dataset/images/val
✓ DataReader initialisé avec 100 images

[4/4] Quantification en cours...
    Format : QDQ (Quantize-DeQuantize)
    Activation : QUInt8
    Weights : QInt8
    Per-channel : Activé



  RÉSULTATS

 Taille des modèles :
    Original (FP32)  :    34.24 MB
    Quantized (INT8) :    11.83 MB
    Réduction        :     2.89x

 Modèle quantifié sauvegardé :
    /workspace/weights/yolop-640-640-quantized-per_channel.onnx

 Nettoyage des fichiers temporaires...
    ✓ Supprimé : temp_simplified.onnx
    ✓ Supprimé : temp_cleaned.onnx
